# Prompting Techniques with Ollama

Prompting steers a pretrained LLM at inference time, with **no weight updates**. In this notebook you will:

1. See four techniques in action: **few-shot**, **chain-of-thought**, **self-consistency**, **RAG**
2. Practice them yourself in short exercises

| Technique | Idea |
|---|---|
| Zero/Few-shot | Give instructions, optionally with examples |
| Chain-of-Thought (CoT) | Ask for step-by-step reasoning before the answer |
| Self-Consistency | Sample several CoT chains, majority-vote the answer |
| RAG | Retrieve relevant documents and put them in the prompt |

## Setup
Run once (Colab with GPU runtime). It installs Ollama and pulls a small model.

In [1]:
%pip install -q ollama scikit-learn numpy
!sudo apt-get install -y -qq zstd pciutils lshw > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1
!nohup ollama serve > ollama.log 2>&1 &
import time; time.sleep(5)
!ollama pull gemma4:e2b

debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 79, <STDIN> line 6.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 



In [9]:
import ollama, re
from collections import Counter

MODEL = "gemma4:e2b"

def generate(prompt, max_tokens=150, temperature=0.0):
    """Send a prompt to the model and return its text. temperature=0 is deterministic."""
    r = ollama.generate(model=MODEL, prompt=prompt, think=False,
                        options={"temperature": temperature, "num_predict": max_tokens})
    return r["response"].strip()

print(generate("Say hello in one short sentence."))

Hello!


## Demo 1: Zero-shot vs. Few-shot
Few-shot adds input→output examples so the model copies the format.

In [10]:
review = "The pacing was dreadful, but the soundtrack was gorgeous."
task = "Classify the review as Positive, Negative, or Mixed. Reply with one word."

zero_shot = f"{task}\n\nReview: {review}\nLabel:"

few_shot = f"""{task}

Review: A masterpiece of modern cinema.
Label: Positive

Review: I fell asleep halfway through.
Label: Negative

Review: Great acting, but the plot made no sense.
Label: Mixed

Review: {review}
Label:"""

print("Zero-shot:", generate(zero_shot, max_tokens=10))
print("Few-shot :", generate(few_shot, max_tokens=10))

Zero-shot: Mixed
Few-shot : Mixed


## Demo 2: Chain-of-Thought
Asking the model to reason step by step helps on multi-step problems.

In [4]:
question = ("A store had 50 shirts. It sold 18 on Monday, got a shipment of 30, "
            "then sold half of what it had. How many shirts are left?")   # answer: 31

print("Direct:\n", generate(f"Q: {question}\nGive only the final number.\nA:", max_tokens=10))
print("\nZero-shot CoT:\n", generate(f"Q: {question}\nA: Let's think step by step.", max_tokens=200))

Direct:
 36

Zero-shot CoT:
 Here is the step-by-step solution:

**1. Starting Inventory:**
The store begins with **50** shirts.

**2. Sales on Monday:**
They sold 18 shirts.
*Shirts remaining after Monday's sales:* $50 - 18 = 32$ shirts.

**3. Shipment Received:**
They received a shipment of 30 shirts.
*Total shirts after the shipment:* $32 + 30 = 62$ shirts.

**4. Second Sale:**
They sold half of what they had at this point (half of 62).
*Shirts sold in the second sale:* $62 / 2 = 31$ shirts.

**5. Final Count:**
*Shirts left:* $62 - 31 = 31$ shirts.

**Answer:** There are **31** shirts left.


## Demo 3: Self-Consistency
Sample several reasoning chains (temperature > 0) and take the most common final answer.

In [5]:
def final_number(text):
    nums = re.findall(r"-?\d+(?:\.\d+)?", text)
    return nums[-1] if nums else None

prompt = (f"Q: {question}\nThink step by step, then end with 'The answer is <number>.'\nA:")
answers = []
for _ in range(5):
    out = generate(prompt, max_tokens=250, temperature=0.8)
    answers.append(final_number(out))

print("Votes :", Counter(answers))
print("Winner:", Counter(answers).most_common(1)[0][0], "(truth: 31)")

Votes : Counter({'31': 5})
Winner: 31 (truth: 31)


## Demo 4: RAG
Retrieve the most relevant document (TF-IDF here; real systems use embeddings + a vector DB) and add it to the prompt.

In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

docs = [
    "Acme Corp's refund policy: items can be returned within 45 days with a receipt.",
    "Acme Corp support hours are 9am-5pm ICT, Monday to Friday.",
    "Acme Corp's Pro plan costs $29 per month and includes 5 seats.",
    "Acme Corp headquarters are located in Phnom Penh, Cambodia.",
]
vec = TfidfVectorizer().fit(docs)
doc_vecs = vec.transform(docs)

def retrieve(query, k=1):
    scores = cosine_similarity(vec.transform([query]), doc_vecs).ravel()
    return [docs[i] for i in scores.argsort()[::-1][:k]]

def rag_answer(query, k=1):
    context = "\n".join(retrieve(query, k))
    return generate(f"Answer using only the context.\n\nContext:\n{context}\n\nQuestion: {query}\nAnswer:")

q = "How many days do I have to return an item at Acme Corp?"
print("No context:", generate(f"Question: {q}\nAnswer:", max_tokens=60))
print("With RAG  :", rag_answer(q))

No context: Please provide me with more information so I can answer your question. I do not have access to specific return policies for Acme Corp.

You will need to check:

* **The Acme Corp website:** Look for their "Returns," "Shipping," or "FAQ" sections.
* **Your
With RAG  : The context does not state how many days you have to return an item at Acme Corp.


---
# Hands-on Practice
Complete the `TODO`s. Re-run each cell and compare outputs. Small models are noisy, so try a few variations.

### Exercise 1: Few-shot extraction
Make the model turn a sentence into JSON with keys `name`, `city`, `age`. First try zero-shot, then add **2 examples** to get a clean, consistent format.

In [ ]:
text = "Sokha, a 29-year-old engineer, just moved to Siem Reap."

# Zero-shot: instruction only
zero_shot = f"""Extract name, city, and age as JSON.

Text: {text}
JSON:"""

# Few-shot: same instruction + 2 worked examples that fix the exact format
# (one line, double quotes, age as a number, no ``` code fences, no extra text)
few_shot = f"""Extract name, city, and age as JSON. Reply with the JSON only.

Text: Dara is 34 and works as a teacher in Battambang.
JSON: {{"name": "Dara", "city": "Battambang", "age": 34}}

Text: Last year, 22-year-old Srey Leak opened a cafe in Kampot.
JSON: {{"name": "Srey Leak", "city": "Kampot", "age": 22}}

Text: {text}
JSON:"""

print("Zero-shot:\n", generate(zero_shot, max_tokens=60))
print("\nFew-shot:\n", generate(few_shot, max_tokens=60))

# Check that the few-shot output is valid JSON with the right keys
import json
out = generate(few_shot, max_tokens=60)
try:
    data = json.loads(out)
    print("\nParsed OK:", data, "| keys:", sorted(data))
except json.JSONDecodeError:
    print("\nNot valid JSON:", out)

**What to look for:** zero-shot usually gets the facts right but wraps the JSON in a ```json code block or changes the layout from run to run. With the two examples, the model copies their format exactly (one line, no code fences), so `json.loads` works and the output can go straight into a program or database. That's the point of few-shot: **examples fix the format**.

### Exercise 2: Chain-of-Thought
The puzzle below often trips up direct answers. Compare a **direct** prompt with a **CoT** prompt. Which is right? (Answer: 0.05, i.e. 5 cents)

In [ ]:
puzzle = "A bat and a ball cost $1.10 in total. The bat costs $1.00 more than the ball. How much is the ball?"

direct = f"Q: {puzzle}\nGive only the final answer in dollars, nothing else.\nA:"
cot    = f"Q: {puzzle}\nA: Let's think step by step. Write out the equations, then end with 'Answer: <number>' (in dollars)."

print("Direct:", generate(direct, max_tokens=40))
print("CoT   :", generate(cot, max_tokens=200))

**What to look for:** the intuitive (wrong) answer is **$0.10**. Working it out: ball = x, bat = x + 1.00, so x + (x + 1.00) = 1.10 → 2x = 0.10 → **x = $0.05**.
The direct prompt forces the model to answer in one go, so it often gives 0.10. The CoT prompt makes it write the equation first, which usually leads to 0.05. If the direct answer happens to be right on your run, that's fine; note it. A model may have seen this famous puzzle during training.

### Exercise 3: Self-Consistency
Write a function `self_consistent(prompt, k)` that samples `k` answers at `temperature=0.8` and returns the majority answer. Test it on the puzzle above with `k=1` vs `k=7`. Does voting help?

*Hint:* ask the model to end with `Answer: <number>` so extracting the answer is easy.

In [ ]:
def extract_answer(text):
    """Pull the final number out of a CoT output. Prefers the 'Answer: <number>' line."""
    m = re.search(r"Answer:\s*\**\s*\$?\s*(-?\d*\.?\d+)\s*(cents?)?", text, re.IGNORECASE)
    if m:
        value = float(m.group(1))
        if m.group(2):          # "5 cents" -> 0.05 dollars
            value /= 100
        return round(value, 2)
    n = final_number(text)      # fallback: last number in the text (helper from Demo 3)
    return round(float(n), 2) if n is not None else None


def self_consistent(prompt, k=5, temperature=0.8, max_tokens=250):
    """Sample k reasoning chains, extract each final answer, return the majority answer."""
    answers = [extract_answer(generate(prompt, max_tokens=max_tokens, temperature=temperature))
               for _ in range(k)]
    votes = Counter(a for a in answers if a is not None)
    print(f"k={k} votes: {dict(votes)}")
    return votes.most_common(1)[0][0] if votes else None


cot_prompt = (f"Q: {puzzle}\nThink step by step, then end with 'Answer: <number>' "
              f"where <number> is the price of the ball in dollars.\nA:")

print("k=1 ->", self_consistent(cot_prompt, k=1), "(truth: 0.05)")
print("k=7 ->", self_consistent(cot_prompt, k=7), "(truth: 0.05)")

**What to look for:** with `k=1` you get a single sample at temperature 0.8, so it can be wrong just by chance (run the cell a few times and you may see 0.1 sometimes). With `k=7`, a single bad chain is outvoted, so the majority answer is much more stable. **Voting helps reliability, but costs k× the tokens and time.** It only helps when the model is right more often than wrong; if most chains make the same mistake, voting just confirms the mistake.

### Exercise 4: RAG
1. Add **two new facts** of your own to `docs` (re-fit the vectorizer after).
2. Ask a question that only your facts can answer.
3. Compare the answer with and without retrieval. What happens when the answer is **not** in the docs? Edit the RAG prompt so the model says *"I don't know"* in that case.

In [ ]:
# 1. Add two new facts of my own, then re-fit the vectorizer
docs = docs + [
    "Acme Corp's annual staff retreat is held in Kampot every December.",
    "Acme Corp's office cat is named Mango and lives at the Phnom Penh headquarters.",
]
vec = TfidfVectorizer().fit(docs)
doc_vecs = vec.transform(docs)   # retrieve() reads these globals, so it now sees the new facts

# 2. A question only my facts can answer
q = "Where is the Acme Corp staff retreat held?"
print("Retrieved :", retrieve(q))
print("No context:", generate(f"Question: {q}\nAnswer:", max_tokens=60))
print("With RAG  :", rag_answer(q))

In [ ]:
# 3. What if the answer is NOT in the docs?
q_missing = "Who is the CEO of Acme Corp?"
print("Retrieved       :", retrieve(q_missing))   # TF-IDF always returns *something*, even if irrelevant
print("Original RAG    :", rag_answer(q_missing))


def rag_answer_safe(query, k=1):
    context = "\n".join(retrieve(query, k))
    prompt = (
        "Answer the question using ONLY the context below.\n"
        "If the context does not contain the answer, reply exactly: I don't know.\n\n"
        f"Context:\n{context}\n\nQuestion: {query}\nAnswer:"
    )
    return generate(prompt, max_tokens=60)


print("Safe RAG (missing):", rag_answer_safe(q_missing))
print("Safe RAG (present):", rag_answer_safe(q))   # should still answer normally

**What to look for:**
- **Without context**, the model can't know a made-up company's retreat location: it either says it doesn't know or invents an answer (hallucination).
- **With RAG**, the retrieved fact is in the prompt, so it should answer "Kampot".
- **When the answer isn't in the docs**, the retriever still hands back the closest doc (here, an unrelated Acme fact), and the original prompt may let the model guess. The safe prompt gives it an explicit way out, so it replies "I don't know" instead of making something up, while still answering normally when the fact *is* there.
- Side note: TF-IDF matches exact words. In Demo 4, the query says "return an item" but the doc says "returned" and "items", so the right doc may not be retrieved; that may be why the saved Demo 4 output says the context doesn't state the answer. Printing `retrieve(q)` like above is a quick way to check. Real systems use embeddings, which match meaning instead of exact words.

### Challenge (optional)
Pick a task from your own work (e.g. classify support tickets, summarize, translate to Khmer). Design a prompt, then improve it by adding **a role, a format constraint, and examples**. Note which change helped most.

In [ ]:
# Task: classify customer support tickets into Billing / Technical / Account / Other
ticket = "I was charged twice for my Pro plan this month, please fix it."

v1 = f"Classify this support ticket.\n\nTicket: {ticket}\nCategory:"

v2 = ("You are a support team lead who routes tickets to the right team.\n"   # + role
      f"Classify this support ticket.\n\nTicket: {ticket}\nCategory:")

v3 = ("You are a support team lead who routes tickets to the right team.\n"
      "Classify the ticket as exactly one of: Billing, Technical, Account, Other. "  # + format constraint
      f"Reply with that one word only.\n\nTicket: {ticket}\nCategory:")

v4 = ("You are a support team lead who routes tickets to the right team.\n"
      "Classify the ticket as exactly one of: Billing, Technical, Account, Other. "
      "Reply with that one word only.\n\n"
      "Ticket: The app crashes every time I open the settings page.\nCategory: Technical\n\n"   # + examples
      "Ticket: How do I change the email address on my profile?\nCategory: Account\n\n"
      "Ticket: My invoice shows the wrong company name.\nCategory: Billing\n\n"
      f"Ticket: {ticket}\nCategory:")

for name, p in [("v1 plain", v1), ("v2 +role", v2), ("v3 +format", v3), ("v4 +examples", v4)]:
    print(f"{name:13}:", generate(p, max_tokens=30))

**What to look for:** v1 and v2 usually give a sentence or a made-up category name. The **format constraint (v3)** is typically the change that helps most, because it limits the answer to four fixed labels. The examples in v4 make it more consistent on unclear tickets. The role mostly changes tone, not accuracy.

---
## Recap
- **Few-shot**: examples fix format and style
- **CoT**: reasoning steps help multi-step problems
- **Self-consistency**: voting across samples improves reliability, at k× the cost
- **RAG**: grounds answers in your own data

Next: when prompting is not enough, **fine-tuning** updates the weights.